# 01 — Exploratory Data Analysis
**SportTalent AI** | Understanding the dataset, feature distributions, and sport profiles.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13})

df = pd.read_csv('../data/combined_sports_data.csv')
print(f'Dataset: {df.shape[0]} rows × {df.shape[1]} columns')
df.head()

In [ ]:
# ── Class distribution ──
fig, ax = plt.subplots(figsize=(9, 4))
counts = df['Sport'].value_counts()
bars = ax.barh(counts.index, counts.values, color=sns.color_palette('muted', len(counts)))
ax.bar_label(bars, padding=3)
ax.set_xlabel('Number of samples')
ax.set_title('Samples per Sport Class')
plt.tight_layout()
plt.savefig('../results/eda_class_dist.png', dpi=150)
plt.show()

In [ ]:
# ── Feature distributions per sport ──
features = ['Endurance', 'Strength', 'Speed', 'Flexibility', 'Cognitive Ability', 'Reflex']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, feat in enumerate(features):
    for sport in df['Sport'].unique():
        data = df[df['Sport'] == sport][feat]
        axes[i].hist(data, bins=15, alpha=0.5, label=sport, density=True)
    axes[i].set_title(feat)
    axes[i].set_xlabel('Score (1-10)')
    if i == 0:
        axes[i].legend(fontsize=7)

plt.suptitle('Feature Distributions by Sport', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../results/eda_feature_distributions.png', dpi=150)
plt.show()

In [ ]:
# ── Sport-level mean radar chart ──
from matplotlib.patches import FancyArrowPatch
import matplotlib.patches as mpatches

sport_means = df.groupby('Sport')[features].mean()
print(sport_means.round(2))

fig, ax = plt.subplots(figsize=(10, 5))
sport_means.T.plot(kind='bar', ax=ax, width=0.7)
ax.set_title('Mean Feature Scores by Sport', fontsize=13)
ax.set_ylabel('Mean Score (1–10)')
ax.set_xticklabels(features, rotation=20, ha='right')
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.savefig('../results/eda_sport_profiles.png', dpi=150)
plt.show()

In [ ]:
# ── Correlation heatmap ──
fig, ax = plt.subplots(figsize=(7, 5))
corr = df[features].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            mask=mask, ax=ax, linewidths=0.5)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('../results/eda_correlation.png', dpi=150)
plt.show()

In [ ]:
# ── PCA 2D scatter ──
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X = StandardScaler().fit_transform(df[features])
pca = PCA(n_components=2, random_state=42)
Xp = pca.fit_transform(X)
print(f'Explained variance: {pca.explained_variance_ratio_.sum():.2%}')

sports = df['Sport'].values
palette = sns.color_palette('tab10', df['Sport'].nunique())
sport_colors = {s: palette[i] for i, s in enumerate(df['Sport'].unique())}

fig, ax = plt.subplots(figsize=(9, 6))
for sport in df['Sport'].unique():
    mask = sports == sport
    ax.scatter(Xp[mask, 0], Xp[mask, 1], label=sport, alpha=0.6, s=25,
               color=sport_colors[sport])
ax.set_title(f'PCA — 2D Projection ({pca.explained_variance_ratio_.sum():.0%} variance)')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.legend()
plt.tight_layout()
plt.savefig('../results/eda_pca.png', dpi=150)
plt.show()